# Orthomosaic — Part 4: PMTiles — Interactive Map Serving

**Tile the COG into a PMTiles archive for interactive map serving — end to end with GeoBrix light-tier readers, writers, and functions (no external tooling).**

1. **`cog_gbx`** reader loads the COG as a raster tile.
2. **`rst_georeference`** reads its pixel scale to pick the native zoom.
3. **`gbx_rst_xyzpyramid`** (a `LATERAL` table UDF) warps it onto the WebMercatorQuad grid on the fly and emits `(z, x, y, PNG bytes)` rows.
4. **`pmtiles_gbx`** writer (`shardZoom=0`) assembles one `.pmtiles` archive directly on the Volume.
5. **`vz.plot_pmtiles`** previews it inline.

> **Prerequisite.** `03_cog` must have written the COG to `cog_dir`.

> **Runtime.** Runs on **Serverless environment 5** (light tier — no JAR, no `rasterio.warp` / `pymbtiles` / `go-pmtiles`).

---

**Last Update:** September 25, 2026

## Setup

In [ ]:
%run ./config_nb

## Step 1: COG → PMTiles (all GeoBrix)

`cog_gbx` reader → `rst_georeference` (native zoom) → `gbx_rst_xyzpyramid` (WebMercator PNG tiles) → `pmtiles_gbx` writer (one `.pmtiles`). Checkpointed per group (skips when the archive already matches the COG signature; `FORCE_PMTILES = True` recomputes).

In [ ]:
import math as _math
import os
import time as _t_pm

from databricks.labs.gbx import pyrx as _pyrx

# gbx_rst_xyzpyramid is a pyrx SQL table UDF (UDTF). config_nb's ds.register installs
# the readers/writers but not the pyrx SQL functions, so register the one UDTF we need
# on THIS session (idempotent; `only=` keeps it to a single spark.udtf.register call).
rx.register(spark, only=["gbx_rst_xyzpyramid"])

# Overview depth below the COG's native zoom, and a hard zoom cap.
_OVERVIEW_LEVELS, _MAX_Z_CAP = 4, 22

_t0 = _t_pm.perf_counter()
try:
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* COGs under {output_dir} — run 01-03 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _mani = _pyrx.Manifest(_gp["checkpoint"])
        _src = _gp["cog"]
        _sig = _pyrx.input_signature({
            "grp": grp,
            "up": (_mani.get_sig("cog", grp) or ""),
            "src_size": (os.path.getsize(_src) if os.path.exists(_src) else 0),
        })
        if _pyrx.checkpoint_skip(_mani, "pmtiles", grp, _sig, force=bool(FORCE_PMTILES)):
            print(f"[pmtiles][skip] group {grp!r} — checkpointed ({_gp['pmtiles']})")
            continue
        # 1. Load the COG through the GeoBrix light COG reader (one whole-raster tile) —
        #    no rasterio, no manual DataFrame construction.
        _cog = spark.read.format("cog_gbx").load(_gp["cog"])
        _cog.createOrReplaceTempView("_cog_tile")
        # 2. Native max-zoom from the tile's OWN georeference (GeoBrix rst_georeference,
        #    EPSG:4326 deg/px). The cos(lat) in the WebMercator resolution and in the
        #    deg→m conversion cancel, so native zoom = log2(1.40625 / scaleX_deg).
        _sx = abs(float(
            _cog.select(rx.rst_georeference("tile").alias("gt")).first()["gt"]["scaleX"]
        ))
        _max_z = max(1, min(_MAX_Z_CAP, int(round(_math.log2(1.40625 / _sx)))))
        _min_z = max(0, _max_z - _OVERVIEW_LEVELS)
        # 3. rst_xyzpyramid (LATERAL UDTF) warps EPSG:4326 → WebMercatorQuad on the fly
        #    and emits (z, x, y, PNG bytes); pmtiles_gbx (shardZoom=0) assembles ONE
        #    .pmtiles archive straight onto the Volume (FUSE-safe sequential write).
        _tiles = spark.sql(
            "SELECT t.z, t.x, t.y, t.bytes FROM _cog_tile, "
            f"LATERAL gbx_rst_xyzpyramid(tile, {_min_z}, {_max_z}, 'PNG', 256, 'bilinear') t"
        )
        (_tiles.write.format("pmtiles_gbx").mode("overwrite")
               .option("shardZoom", "0").save(_gp["pmtiles"]))
        _mani.mark_done("pmtiles", grp, _sig, _gp["pmtiles"])
        print(f"  group {grp!r}: z{_min_z}-{_max_z} → PMTiles → {_gp['pmtiles']}")
    print(f"PMTiles for {len(_groups)} group(s) in {_t_pm.perf_counter()-_t0:.1f}s")
except Exception as e:
    print(f"[ERROR] PMTiles step failed after {_t_pm.perf_counter()-_t0:.1f}s: {e}")
    raise

## Step 2: In-notebook Preview

In [ ]:
vz.plot_pmtiles(group_paths(discover_groups()[0])["pmtiles"])

## Steps performed

1. **Load COG** — the GeoBrix `cog_gbx` reader loaded the COG as a raster tile.
2. **Native zoom** — `rst_georeference` gave the pixel scale; the WebMercator native zoom is `log2(1.40625 / scaleX)` (the cos-latitude terms cancel), with a few overview levels below it.
3. **XYZ pyramid** — `gbx_rst_xyzpyramid` warped EPSG:4326 → WebMercatorQuad and emitted `(z, x, y, PNG)` tiles.
4. **PMTiles archive** — the `pmtiles_gbx` writer assembled a single `.pmtiles` file on the Volume.
5. **Preview** — `vz.plot_pmtiles` rendered the raster tiles inline over a no-key basemap.

**Series complete.** The orthomosaic GeoTIFF, COG, and PMTiles are available in `output_dir` and the configured UC Volume.

**Re-run behaviour:** the PMTiles step is checkpointed per group. A re-run skips any group whose archive already matches the COG signature. Set `FORCE_PMTILES = True` in `config_nb` to recompute.